# Классическая латынь как сравнительный фон

Вспомогательный расчёт для тех же **десяти терминов власти**: пятнадцать ближайших соседей в *Opera Latina* и *Opera Maiora*. Данные двух моделей **не выравниваются** и их косинусные значения не интерпретируются как межэпохальные расстояния. Таблица предназначена для сопоставления локальных окружений.

Выполните ячейки по порядку. Обе опубликованные бинарные модели проверяются по SHA-256; они не включены в репозиторий. Для лемм с частотой менее 50 употреблений требуется особая осторожность.

## 1. Загрузка моделей и параметры

In [3]:
import importlib.util, subprocess, sys
if importlib.util.find_spec("gensim") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "gensim"])

import hashlib, os, urllib.request
import pandas as pd
from gensim.models import KeyedVectors
from gensim.models.fasttext import load_facebook_model
from IPython.display import display, Markdown

# --- Папки репозитория ---------------------------------------------------------
# Ноутбуки лежат в папке notebooks/. Модели загружаются в папку models/, рисунки
# сохраняются в папку figures/ (в Colab – папки рабочего каталога).
ROOT = ".." if os.path.basename(os.getcwd()) == "notebooks" else "."
MODEL_DIR, FIG_DIR = os.path.join(ROOT, "models"), os.path.join(ROOT, "figures")
os.makedirs(MODEL_DIR, exist_ok=True)

POWER_TERMS = ["dominium", "potestas", "praelatio", "iurisdictio", "auctoritas",
               "imperium", "principatus", "regimen", "gubernatio", "regnum"]
TOP = 15
RARE = 50          # реже этого числа употреблений вектор считается ненадёжным

MODELS = {
    "Opera Latina": {"file": os.path.join(MODEL_DIR, "allLASLA-lemmi-fast-100-SKIP-win5-min5.bin"),
                     "url": "https://embeddings.lila-erc.eu/samples/download/fasttext/allLASLA-lemmi-fast-100-SKIP-win5-min5.bin",
                     "sha256": "33d38b9564ae1cbc32c2d88e506efe9d34a89cc490db9fd4962720245ac94a9c"},
    "Opera Maiora": {"file": os.path.join(MODEL_DIR, "opera-maiora-lemmas_skip_100.bin"),
                     "url": "https://embeddings.lila-erc.eu/samples/download/OperaMaiora/opera-maiora-lemmas_skip_100.bin",
                     "sha256": "4f973ddb57a9a382b13345c72269c628f10bae1b4b279f26dca9168697b44f5c"},
}


def load_lila_model(info):
    # Загружает двоичную модель fastText проекта LiLa и возвращает векторы её словаря
    # (с частотами лемм) и параметры обучения. Работаем только со словарём модели:
    # для слов вне словаря fastText строит векторы из частей слова, такие векторы не используются.
    if not os.path.exists(info["file"]):
        print(f"Загружается {os.path.basename(info['file'])} (около 770 МБ)…")
        urllib.request.urlretrieve(info["url"], info["file"])
    digest = hashlib.sha256()
    with open(info["file"], "rb") as f:
        for chunk in iter(lambda: f.read(1 << 24), b""):
            digest.update(chunk)
    if info.get("sha256") and digest.hexdigest() != info["sha256"]:
        raise ValueError("Контрольная сумма не совпадает: это другая версия модели.")
    print(f"{os.path.basename(info['file'])}: SHA-256 {digest.hexdigest()}")

    model = load_facebook_model(info["file"])
    keys = model.wv.index_to_key
    kv = KeyedVectors(vector_size=model.wv.vector_size)
    kv.add_vectors(keys, model.wv.vectors[:len(keys)])
    for k in keys:
        kv.set_vecattr(k, "count", model.wv.get_vecattr(k, "count"))
    params = {"Архитектура": "skip-gram" if model.sg else "CBOW",
              "Размерность векторов": model.wv.vector_size, "Контекстное окно": model.window,
              "Минимальная частота": model.min_count, "Эпох обучения": model.epochs,
              "Отрицательных примеров": model.negative,
              "Длина n-грамм символов": f"{model.wv.min_n}–{model.wv.max_n}",
              "Лемм в словаре": len(keys),
              "Словоупотреблений в корпусе": f"{model.corpus_total_words:,}".replace(",", " ")}
    del model
    return kv, params


KV, PARAMS = {}, {}
for name, info in MODELS.items():
    KV[name], PARAMS[name] = load_lila_model(info)
display(pd.DataFrame(PARAMS))

allLASLA-lemmi-fast-100-SKIP-win5-min5.bin: SHA-256 33d38b9564ae1cbc32c2d88e506efe9d34a89cc490db9fd4962720245ac94a9c
opera-maiora-lemmas_skip_100.bin: SHA-256 4f973ddb57a9a382b13345c72269c628f10bae1b4b279f26dca9168697b44f5c


,Opera Latina,Opera Maiora
Архитектура,skip-gram,skip-gram
Размерность векторов,100,100
Контекстное окно,10,10
Минимальная частота,5,5
Эпох обучения,5,5
Отрицательных примеров,25,25
Длина n-грамм символов,3–6,3–6
Лемм в словаре,11327,6724
Словоупотреблений в корпусе,1 911 329,4 346 955


## 2. Пятнадцать ближайших соседей для каждого термина

In [4]:
def cell(name, term):
    kv = KV[name]
    if term not in kv.key_to_index:
        return "–", "нет в словаре модели"
    n = kv.get_vecattr(term, "count")
    mark = " (вектор ненадёжен)" if n < RARE else ""
    return n, ", ".join(f"{w} {s:.3f}" for w, s in kv.most_similar(term, topn=TOP)) + mark

rows = []
for term in POWER_TERMS:
    f_ol, n_ol = cell("Opera Latina", term)
    f_om, n_om = cell("Opera Maiora", term)
    rows.append({"Лемма": term, "Частота OL": f_ol, "Соседи в Opera Latina": n_ol,
                 "Частота OM": f_om, "Соседи в Opera Maiora": n_om})
table = pd.DataFrame(rows).set_index("Лемма").astype(str)
display(Markdown(f"**{TOP} ближайших соседей в двух моделях (косинусное сходство)**"))
display(table.style.set_properties(**{"text-align": "left", "white-space": "normal"}))

# Машиночитаемая выгрузка сравнительной таблицы.
TBL_DIR = os.path.join(ROOT, "tables")
os.makedirs(TBL_DIR, exist_ok=True)
out_path = os.path.join(TBL_DIR, "classical_neighbors.csv")
table.to_csv(out_path, encoding="utf-8-sig")
print("Таблица сохранена:", out_path)


**15 ближайших соседей в двух моделях (косинусное сходство)**

,Частота OL,Соседи в Opera Latina,Частота OM,Соседи в Opera Maiora
Лемма,,,,
dominium,12,"lenocinium 0.763, minium 0.737, dominus 0.693, dominor 0.691, alienum 0.660, infortunium 0.658, munditia 0.656, sordide 0.654, peruagatus 0.653, domicilium 0.638, pellicula 0.635, dominans 0.635, domina 0.634, indigens 0.630, abiectus 0.629 (вектор ненадёжен)",287,"seruitus 0.684, seruus 0.682, dominor 0.658, coerceo 0.633, potestas 0.632, seruitium 0.615, dominatio 0.611, conseruus 0.599, despoticus 0.595, usurpo 0.593, iurisconsultus 0.593, mercatio 0.584, tyrannicus 0.572, patrimonium 0.570, latrocinium 0.568"
potestas,668,"auctionor 0.654, contionor 0.653, exceptio 0.649, obnuntio 0.649, adiudico 0.642, intestatus 0.642, obtestatio 0.640, abiudico 0.638, enuntio 0.635, magistratus 0.623, testatus 0.615, denuntio 0.613, iniquitas 0.612, extraordinarius 0.612, imperium 0.611",2527,"iurisdictio 0.665, eximo 0.645, minister 0.643, dominium 0.632, ecclesia 0.628, possum 0.624, episcopatus 0.622, arbiter 0.620, resigno 0.620, sacerdotalis 0.620, ordinatio 0.617, ordinarius 0.617, regalis 0.616, inuincibilis 0.615, iudiciarius 0.614"
praelatio,–,нет в словаре модели,249,"celatio 0.734, subiectio 0.712, praefero 0.671, uelatio 0.657, plebeius 0.657, ordinatio 0.655, praeficio 0.655, praepono 0.654, ordinarius 0.648, tyrannicus 0.644, officium 0.638, primatus 0.634, procuratio 0.632, is 0.631, episcopatus 0.628"
iurisdictio,18,"dictio 0.878, decemuiralis 0.821, iurisconsultus 0.820, duumuir 0.794, rogatio 0.785, sortitio 0.783, decemuir 0.781, priuilegium 0.778, exactio 0.774, curiatus 0.773, praeiudicium 0.772, dictito 0.766, tactio 0.760, intercessio 0.760, postulatio 0.756 (вектор ненадёжен)",97,"degrado 0.761, excommunicatio 0.726, clauis 0.720, forum 0.711, episcopus 0.699, laicus 0.698, absolutio 0.696, parochialis 0.679, subdiaconatus 0.672, indulgentia 0.669, ordinarius 0.669, excommunico 0.667, potestas 0.665, diaconatus 0.662, uicarius 0.660"
auctoritas,789,"integritas 0.696, consensio 0.691, grauitas 0.675, exauctoro 0.672, repudio 0.671, aucto 0.667, existimatio 0.665, rogatio 0.659, obtempero 0.654, prauitas 0.652, archytas 0.651, emblema 0.649, consensus 0.648, disceptatio 0.647, constantia 0.646",1363,"subauctoritas 0.870, auctoritatiuus 0.772, induco 0.695, assertiuus 0.671, solutio 0.668, apostolicus 0.662, declaratio 0.659, tractator 0.654, obiectio 0.653, expositio 0.653, probatio 0.652, ceterum 0.649, aio 0.645, improbatio 0.642, auctor 0.640"
imperium,1325,"imperito 0.705, imperitia 0.704, maroboduus 0.700, antiochia 0.694, seruitium 0.694, externi 0.688, arsacidae 0.687, abdagaeses 0.685, micipsa 0.684, imperite 0.682, puerperium 0.676, syphax 0.676, iudaei 0.674, regnum 0.672, uonones 0.670",449,"impero 0.794, imperatiuus 0.727, obedio 0.724, despoticus 0.718, exequor 0.695, nutus 0.646, imperitus 0.639, motiuus 0.631, subiaceo 0.630, regalis 0.616, elicitiuus 0.607, persuasibilis 0.594, uis 0.583, potestas 0.580, ordinatio 0.579"
principatus,96,"principalis 0.867, initium 0.757, principium 0.738, princeps 0.683, obtineo 0.667, peruagatus 0.660, hiempsal 0.656, praecello 0.649, coitio 0.648, consultatio 0.647, citerior 0.646, transalpinus 0.646, praepotens 0.645, primores 0.642, etiamtum 0.640",160,"dominatio 0.788, principor 0.784, princeps 0.758, despoticus 0.747, prouincia 0.739, archangelus 0.729, dominatiuus 0.720, regalis 0.708, optimas 0.701, aristocratia 0.678, michael 0.665, regnatiuus 0.653, regimen 0.646, tyrannicus 0.631, praesideo 0.629"
regimen,24,"tegimen 0.762, specimen 0.690, leuamen 0.641, conformo 0.637, inanimus 0.636, quonam 0.634, rego 0.633, prorsus 0.628, informo 0.628, magnanimus 0.624, augmen 0.619, intentatus 0.619, inconsultus 0.611, proiectus 0.609, prorsum 0.609 (вектор ненадёжен)",189,"praeficio 0.738, gubernatiuus 0.713, gubernatio 0.705, aristocratia 0.703, optimas 0.696, regitiuus 0.686, ciuitas 0.683, rector 0.668, tyrannicus 0.668, regius 0.665, subdo 0.665, ordinator 0.664, regalis 0.

Таблица сохранена: ..\tables\classical_neighbors.csv


## Ограничения

Результаты относятся только к опубликованным моделям и отображают дистрибутивное сходство. Корпуса различаются составом и объёмом, модельные пространства не выравнивались. Для более редких лемм ближайших соседей следует интерпретировать осторожно.